<a href="https://colab.research.google.com/github/Noadori34/Project-data-science/blob/main/notebooks/02_basic_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Basic CNN with residual blocks (kidney ultrasound)

**Goal:** classify kidney ultrasound images as healthy vs. pathological using a small CNN trained from scratch, inspired by ResNet (We use residual blocks (as in ResNet) because they are a simple, well-tested way to build a CNN that trains reliably. Skip connections help gradients flow, so even a small network learns well. It also makes our from-scratch model directly comparable to a pretrained ResNet later).

**Pipeline:**
1. Setup and data inventory
2. Build the dataset (labels + train/val/test split)
3. Define the CNN
4. Train and evaluate

In [1]:
# 1. SETUP

import os

REPO_URL = "https://github.com/Noadori34/Project-data-science.git"
REPO_DIR = "/content/Project-data-science"

# Clone the repo so the notebook works in anyone's Colab
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

%cd {REPO_DIR}
!ls

Cloning into 'Project-data-science'...
remote: Enumerating objects: 8264, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 8264 (delta 11), reused 27 (delta 4), pack-reused 8218 (from 1)
Receiving objects: 100% (8264/8264), 198.53 MiB | 20.89 MiB/s, done.
Resolving deltas: 100% (556/556), done.
Updating files: 100% (8680/8680), done.
/content/Project-data-science
bbdd_kidney	Documents		      kidneyUS-main  results
description.md	kidneyUS_images_25_june_2025  notebooks


In [2]:
# Inspect the data folders: list what each dataset contains
# (to find out which files/labels we have before building the dataset)
!ls bbdd_kidney
!ls kidneyUS-main | head -20
!ls kidneyUS_images_25_june_2025 | head -20

images	labels	masks
annotation_guidelines.docx
Datasheet for Data Set.pdf
kidneyUSlogo.png
labels
OpenKidneyUltrasoundDataSet_TransducerInfo.xlsx
README.md
requirements.txt
src
100_IM-0246-0030_anon.png
100_IM-0246-0082_anon.png
101_IM-0027-0028_anon.png
102_IM-0262-0009_anon.png
102_IM-0262-0010_anon.png
103_IM-0714-0081_anon.png
104_IM-0112-0030_anon.png
104_IM-0112-0078_anon.png
105_IM-0418-0051_anon.png
106_IM-0451-0008_anon.png
107_IM-0250-0022_anon.png
108_IM-0831-0005_anon.png
10_IM-0415-0078_anon.png
110_IM-0026-0001_anon.png
110_IM-0026-0005_anon.png
111_IM-0490-0016_anon.png
112_IM-0503-0038_anon.png
113_IM-0536-0008_anon.png
115_IM-0474-0001-0002_anon.png
116_IM-0407-0048_anon.png


In [3]:
# Count files in each folder and peek at the labels of bbdd_kidney
!echo "images:" && ls bbdd_kidney/images | wc -l
!echo "labels:" && ls bbdd_kidney/labels | wc -l
!echo "masks:" && ls bbdd_kidney/masks | wc -l
!echo "Open Kidney images:" && ls kidneyUS_images_25_june_2025 | wc -l

!ls bbdd_kidney/labels | head -10
!ls bbdd_kidney/images | head -10

images:
1985
labels:
1985
masks:
1985
Open Kidney images:
536
1000.txt
1001.txt
1002.txt
1003.txt
1004.txt
1005.txt
1006.txt
1007.txt
1008.txt
1009.txt
1000.jpg
1001.jpg
1002.jpg
1003.jpg
1004.jpg
1005.jpg
1006.jpg
1007.jpg
1008.jpg
1009.jpg


In [4]:
# Look at the content of some label files and check masks
import glob, collections

label_files = sorted(glob.glob("bbdd_kidney/labels/*.txt"))

# Print the first 5 label files
for f in label_files[:5]:
    print(f, "->", repr(open(f).read()))

# How many label files are empty?
empty = sum(1 for f in label_files if os.path.getsize(f) == 0)
print("Empty label files:", empty, "of", len(label_files))

# Which class values appear in the first column?
classes = collections.Counter()
for f in label_files:
    for line in open(f).read().strip().splitlines():
        classes[line.split()[0]] += 1
print("Class values:", classes)

!ls bbdd_kidney/masks | head -5

bbdd_kidney/labels/1.txt -> 'Image filename: 1.jpg\nGlobal diagnosis: 1\nGlobal pathologies: 0 0\nView: 0\nBounding box 1: 326 196 48 32 5\n'
bbdd_kidney/labels/10.txt -> 'Image filename: 10.jpg\nGlobal diagnosis: 1\nGlobal pathologies: 1 0\nView: 0\n'
bbdd_kidney/labels/100.txt -> 'Image filename: 100.jpg\nGlobal diagnosis: 1\nGlobal pathologies: 0 0\nView: 0\nBounding box 1: 325 180 35 72 1\nBounding box 2: 224 220 46 115 8\n'
bbdd_kidney/labels/1000.txt -> 'Image filename: 1000.jpg\nGlobal diagnosis: 1\nGlobal pathologies: 0 0\nView: 0\nBounding box 1: 375 168 80 58 7\nBounding box 2: 286 182 142 83 7\nBounding box 3: 391 133 39 27 8\nBounding box 4: 415 139 23 31 8\n'
bbdd_kidney/labels/1001.txt -> 'Image filename: 1001.jpg\nGlobal diagnosis: 1\nGlobal pathologies: 1 1\nView: 0\nBounding box 1: 384 194 46 40 2\nBounding box 2: 311 208 44 33 2\n'
Empty label files: 0 of 1985
Class values: Counter({'Global': 3970, 'Bounding': 2478, 'Image': 1985, 'View:': 1985})
1000.mat
1001.mat
100